# IARA-Imagem — GRANDE BATERIA (~8h de GPU)

Base: a pirâmide já **bate o JPEG** em dados diversos (Δ +3.13 dB médio) depois do conserto do loss
progressivo. Esta bateria ataca as duas frentes que faltam:

**A) Bater os codecs MODERNOS** — JPEG é barra baixa. Aqui entram **WebP, AVIF e PNG**.

**B) Devolver o GRÃO** — o resultado parece "pincel" porque MSE devolve a MÉDIA das texturas plausíveis.
Estágio 2 (LPIPS + GAN leve no decoder) devolve a textura. Reportamos os **dois eixos** (PSNR cai,
realismo sobe — é o trade-off taxa-distorção-percepção, e é lei, não bug).

| exp | pergunta |
|---|---|
| 1 | está sub-treinado? (15k → 40k → 80k passos) |
| 2 | curva R-D honesta vs **JPEG/WebP/AVIF/PNG** (varredura de λ) |
| 3 | modelo maior ajuda? (5 graus / mais canais) |
| 4 | **o grão volta?** (LPIPS + GAN no decoder, 2 eixos) |
| 5 | domínio estreito vs diverso |

Tudo com checkpoint + `resultados.json` incremental (sobrevive a desconexão).
**Rode com GPU.** Se cair, é só executar tudo de novo — ele pula o que já fez.


## 1) Setup, dados e os BASELINES modernos


In [ ]:
import subprocess, sys, os
def pipq(*p): subprocess.run([sys.executable,"-m","pip","-q","install",*p])
pipq("lpips","pillow-avif-plugin")
import torch, time, glob, json, io, math
import numpy as np, torch.nn as nn, torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
try:
    import pillow_avif  # registra AVIF no PIL
    HAS_AVIF=True
except Exception: HAS_AVIF=False
try:
    import lpips as _lp; HAS_LPIPS=True
except Exception: HAS_LPIPS=False
assert torch.cuda.is_available(), "Ligue a GPU!"
print("GPU:",torch.cuda.get_device_name(0),"| AVIF:",HAS_AVIF,"| LPIPS:",HAS_LPIPS)
T0=time.time(); OUT="/content/out"; os.makedirs(OUT,exist_ok=True)
def log(m): print(f"[{(time.time()-T0)/60:6.1f}min] {m}",flush=True)
DATA="/content/val2017"
if not os.path.isdir(DATA):
    log("baixando COCO val2017 …")
    subprocess.run(["wget","-q","http://images.cocodataset.org/zips/val2017.zip","-O","/content/coco.zip"],check=True)
    subprocess.run(["unzip","-q","/content/coco.zip","-d","/content/"],check=True)
    os.remove("/content/coco.zip")
FILES=sorted(glob.glob(f"{DATA}/*.jpg")); log(f"{len(FILES)} imagens")
RES=json.load(open(f"{OUT}/resultados.json")) if os.path.exists(f"{OUT}/resultados.json") else {}
def save(): json.dump(RES,open(f"{OUT}/resultados.json","w"),indent=1)
log(f"experimentos já feitos: {list(RES.keys())}")


## 2) Arquitetura + loss progressivo (a correção que virou o jogo)


In [ ]:
LN2=float(np.log(2)); S=128; dev="cuda"
def chans(d,c): return [3]+[32,48,64,96,128][:d-1]+[c]
def enc_stack(cs):
    L=[]
    for i in range(len(cs)-1): L+=[nn.Conv2d(cs[i],cs[i+1],4,2,1),nn.GELU()]
    return nn.Sequential(*L[:-1])
def dec_stack(cs):
    L=[]
    for i in range(len(cs)-1): L+=[nn.ConvTranspose2d(cs[i],cs[i+1],4,2,1),nn.GELU()]
    return nn.Sequential(*L[:-1])
class PyramidN(nn.Module):
    def __init__(self,specs):
        super().__init__(); self.specs=specs
        self.encs=nn.ModuleList([enc_stack(chans(d,c)) for d,c in specs])
        self.decs=nn.ModuleList([dec_stack(list(reversed(chans(d,c)))) for d,c in specs])
        self.logb=nn.ParameterList([nn.Parameter(torch.zeros(c)) for _,c in specs])
    def _q(self,z,train): return z+(torch.rand_like(z)-0.5) if train else torch.round(z)
    def forward(self,x,train=True):
        rec=None; zs=[]; recs=[]
        for i in range(len(self.specs)):
            tgt=x if rec is None else (x-rec)
            z=self._q(self.encs[i](tgt),train); zs.append(z)
            out=self.decs[i](z)
            rec=torch.sigmoid(out) if i==0 else (rec+out).clamp(0,1)
            recs.append(rec)
        return {"rec":rec,"recs":recs,"z":zs}
def rate(z,logb):
    b=F.softplus(logb)[None,:,None,None]+1e-3
    return (torch.log(2*b)+z.abs()/b).mean()/LN2
def kb_of(z):
    v=z.detach().cpu().numpy().ravel().astype(np.int32)
    if v.size==0: return 0.0
    p=np.bincount(v-v.min()).astype(np.float64); p=p[p>0]/v.size
    return v.size*float(-(p*np.log2(p)).sum())/8/1024
def psnr(a,b):
    m=float(((a-b)**2).mean()); return 99.0 if m<1e-12 else 10*np.log10(1.0/m)
def sharp(img):   # nitidez = variância do laplaciano (a métrica do M96)
    g=img.mean(0) if img.ndim==3 else img
    k=torch.tensor([[0,1,0],[1,-4,1],[0,1,0]],dtype=torch.float32)[None,None]
    return float(F.conv2d(torch.as_tensor(g)[None,None],k).var())
BASE_SPECS=[(5,24),(4,16),(3,12),(2,8)]; BIG_SPECS=[(6,32),(5,24),(4,16),(3,12),(2,8)]
MULT=2**max(d for d,_ in BIG_SPECS)
print("MULT (múltiplo exigido dos lados):",MULT)


## 3) Dados na GPU + BASELINES (JPEG · WebP · AVIF · PNG)


In [ ]:
N_IMGS=3000; HELD=64
def load(fs,cap=320):
    out=[]
    for f in fs:
        try:
            im=Image.open(f).convert("RGB")
            if min(im.size)<S: continue
            if min(im.size)>cap: r=cap/min(im.size); im=im.resize((int(im.width*r),int(im.height*r)))
            w,h=im.size; im=im.crop((0,0,w//MULT*MULT,h//MULT*MULT))
            if min(im.size)<S: continue
            out.append(torch.from_numpy(np.asarray(im)).float().permute(2,0,1)/255.)
        except Exception: pass
    return out
ALL=load(FILES[:N_IMGS]); held=ALL[-HELD:]; pool=[t.to(dev) for t in ALL[:-HELD]]
log(f"treino {len(pool)} · held-out {len(held)}")
def crops(idxs,n):
    out=[]
    for _ in range(n):
        im=pool[idxs[int(torch.randint(0,len(idxs),(1,)))]]
        i=int(torch.randint(0,im.shape[1]-S+1,(1,))); j=int(torch.randint(0,im.shape[2]-S+1,(1,)))
        out.append(im[:,i:i+S,j:j+S])
    return torch.stack(out)
# ---- baselines: varre a curva de cada codec UMA vez por imagem (cacheado) ----
_BL={}
def baselines(img_t,key):
    if key in _BL: return _BL[key]
    arr=(img_t.permute(1,2,0).cpu().numpy()*255).astype(np.uint8); ref=arr.astype(np.float32)/255
    im=Image.fromarray(arr); d={}
    for name,fmt,qs in [("JPEG","JPEG",range(2,98,3)),("WebP","WEBP",range(2,98,3))]+([("AVIF","AVIF",range(5,96,6))] if HAS_AVIF else []):
        pts=[]
        for q in qs:
            try:
                b=io.BytesIO(); im.save(b,fmt,quality=int(q))
                rec=np.asarray(Image.open(io.BytesIO(b.getvalue())).convert("RGB"),np.float32)/255
                pts.append((len(b.getvalue())/1024,psnr(rec,ref)))
            except Exception: pass
        if pts: d[name]=pts
    b=io.BytesIO(); im.save(b,"PNG",optimize=True); d["PNG_lossless_KB"]=len(b.getvalue())/1024
    _BL[key]=d; return d
def bl_at(img_t,key,kb):
    d=baselines(img_t,key)
    return {n:min(p,key=lambda t:abs(t[0]-kb))[1] for n,p in d.items() if n!="PNG_lossless_KB"}
log("baselines prontos: JPEG, WebP"+(", AVIF" if HAS_AVIF else " (AVIF indisponível)")+", PNG(lossless)")


## 4) Treinador + avaliador (todos os codecs de uma vez)


In [ ]:
def treinar(steps,lam,specs=None,idxs=None,B=32,lr=6e-4,tag="run",every=5000):
    specs=specs or BASE_SPECS; idxs=idxs or list(range(len(pool)))
    ck=f"{OUT}/{tag}.pt"
    m=PyramidN(specs).to(dev)
    if os.path.exists(ck):
        m.load_state_dict(torch.load(ck)); m.eval(); log(f"  [{tag}] checkpoint reaproveitado"); return m
    opt=torch.optim.AdamW(m.parameters(),lr=lr)
    sch=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=lr,total_steps=steps,pct_start=0.05)
    hist=[]
    for s in range(steps):
        x=crops(idxs,B); o=m(x,train=True)
        loss=sum(F.mse_loss(r,x) for r in o["recs"])+lam*sum(rate(z,lb) for z,lb in zip(o["z"],m.logb))
        opt.zero_grad(); loss.backward(); opt.step(); sch.step(); hist.append(loss.item())
        if s%every==0 or s==steps-1:
            with torch.no_grad(): kbs=[round(kb_of(z),2) for z in m(x,train=False)["z"]]
            log(f"  [{tag}] {s}/{steps} loss {np.mean(hist[-200:]):.4f} | KB/grau {kbs}")
    torch.save(m.state_dict(),ck); m.eval(); return m
def avaliar(m,imgs,n=24,tag=""):
    rows=[]; nl=len(m.specs)
    with torch.no_grad():
        for L in range(1,nl+1):
            ps,kbs,sh,bl=[],[],[],{}
            for i,im in enumerate(imgs[:n]):
                x=im[None].to(dev); o=m(x,train=False)
                rec=o["recs"][L-1][0].cpu(); kb=sum(kb_of(z) for z in o["z"][:L])
                ps.append(psnr(rec.numpy(),x[0].cpu().numpy())); kbs.append(kb); sh.append(sharp(rec))
                for k2,v in bl_at(im,i,kb).items(): bl.setdefault(k2,[]).append(v)
            r={"nivel":L,"kb":float(np.mean(kbs)),"psnr":float(np.mean(ps)),"sharp":float(np.mean(sh))}
            r.update({k2:float(np.mean(v)) for k2,v in bl.items()}); rows.append(r)
    if tag:
        f=rows[-1]; comp=" · ".join(f"{k2} {f[k2]:.2f} (Δ{f['psnr']-f[k2]:+.2f})" for k2 in f if k2 in ("JPEG","WebP","AVIF"))
        log(f"  [{tag}] nível{f['nivel']}: {f['psnr']:.2f} dB @ {f['kb']:.2f} KB | {comp}")
    return rows
PNG_KB=float(np.mean([baselines(im,i)["PNG_lossless_KB"] for i,im in enumerate(held[:24])]))
log(f"PNG sem perdas: {PNG_KB:.1f} KB/imagem (referência de compressão SEM perda)")


## 5) EXP-1 — está sub-treinado? (15k → 40k → 80k)


In [ ]:
for st in [15000,40000,80000]:
    k=f"steps{st}"
    if k in RES: log(f"EXP1 {k}: já feito"); continue
    m=treinar(st,1.2e-3,tag=k); RES[k]=avaliar(m,held,tag=k); save()
xs=[15000,40000,80000]; ys=[RES[f"steps{s}"][-1]["psnr"] for s in xs]
plt.figure(figsize=(5,3)); plt.plot(xs,ys,"o-"); plt.xlabel("passos"); plt.ylabel("PSNR nível final")
plt.title("ainda sobe = era sub-treino"); plt.grid(alpha=.3); plt.show()
print("ganho 15k→80k:",round(ys[-1]-ys[0],2),"dB")


## 6) EXP-2 — 🏆 curva R-D vs JPEG · WebP · AVIF (varredura de λ)


In [ ]:
BEST=80000
LAMS=[3e-4,1.2e-3,4e-3,1.2e-2]
for lam in LAMS:
    k=f"lam{lam:.0e}"
    if k in RES: log(f"EXP2 {k}: já feito"); continue
    m=treinar(BEST,lam,tag=k); RES[k]=avaliar(m,held,tag=k); save()
pts=sorted([(RES[f"lam{l:.0e}"][-1]) for l in LAMS],key=lambda r:r["kb"])
plt.figure(figsize=(7,4.5))
plt.plot([p["kb"] for p in pts],[p["psnr"] for p in pts],"o-",lw=2,label="IARA pirâmide")
for nm,st in [("JPEG","s--"),("WebP","^--"),("AVIF","d--")]:
    if nm in pts[0]: plt.plot([p["kb"] for p in pts],[p[nm] for p in pts],st,label=nm)
plt.xlabel("KB"); plt.ylabel("PSNR (dB)"); plt.legend(); plt.grid(alpha=.3)
plt.title(f"curva R-D (PNG sem perdas = {PNG_KB:.0f} KB)"); plt.show()
for nm in ("JPEG","WebP","AVIF"):
    if nm in pts[0]: print(f"Δ médio vs {nm}: {np.mean([p['psnr']-p[nm] for p in pts]):+.2f} dB")
print(f"compressão vs PNG sem perdas: {PNG_KB/np.mean([p['kb'] for p in pts]):.1f}×")


## 7) EXP-3 — modelo MAIOR ajuda? (5 graus, mais canais)


In [ ]:
if "grande" not in RES:
    m=treinar(BEST,1.2e-3,specs=BIG_SPECS,tag="grande"); RES["grande"]=avaliar(m,held,tag="grande"); save()
a=RES["lam1e-03"][-1]; b=RES["grande"][-1]
print(f"base (4 graus): {a['psnr']:.2f} dB @ {a['kb']:.2f} KB")
print(f"grande(5 graus): {b['psnr']:.2f} dB @ {b['kb']:.2f} KB")


## 8) EXP-4 — 🎨 DEVOLVER O GRÃO (LPIPS + GAN no decoder)

MSE devolve a **média** das texturas plausíveis → "pincel". Aqui congelamos o encoder (taxa NÃO muda)
e refinamos só o decoder com perda perceptual + adversarial. **PSNR vai cair, realismo vai subir** —
isso é o trade-off taxa-distorção-percepção, é lei. Reportamos os dois eixos e as imagens.


In [ ]:
class Disc(nn.Module):
    def __init__(s):
        super().__init__()
        ch=[3,64,128,256]; L=[]
        for i in range(3): L+=[nn.Conv2d(ch[i],ch[i+1],4,2,1),nn.LeakyReLU(0.2,True)]
        L+=[nn.Conv2d(256,1,3,1,1)]; s.net=nn.Sequential(*L)
    def forward(s,x): return s.net(x)
def estagio2(base_tag="lam1e-03",steps=12000,w_lpips=1.0,w_gan=0.02,tag="grao"):
    ck=f"{OUT}/{tag}.pt"
    m=PyramidN(BASE_SPECS).to(dev); m.load_state_dict(torch.load(f"{OUT}/{base_tag}.pt"))
    if os.path.exists(ck): m.load_state_dict(torch.load(ck)); m.eval(); log(f"[{tag}] checkpoint"); return m
    for p in m.encs.parameters(): p.requires_grad_(False)      # ENCODER congelado → taxa não muda
    for p in m.logb.parameters(): p.requires_grad_(False)
    D=Disc().to(dev)
    oG=torch.optim.AdamW([p for p in m.decs.parameters()],lr=1e-4)
    oD=torch.optim.AdamW(D.parameters(),lr=1e-4)
    lp=_lp.LPIPS(net="vgg").to(dev) if HAS_LPIPS else None
    for p in (lp.parameters() if lp else []): p.requires_grad_(False)
    for s in range(steps):
        x=crops(list(range(len(pool))),16); o=m(x,train=True); r=o["rec"]
        # D
        dl=(F.relu(1-D(x)).mean()+F.relu(1+D(r.detach())).mean())
        oD.zero_grad(); dl.backward(); oD.step()
        # G: fidelidade + percepção + realismo
        o=m(x,train=True); r=o["rec"]
        gl=F.mse_loss(r,x)
        if lp is not None: gl=gl+w_lpips*lp(r*2-1,x*2-1).mean()
        gl=gl-w_gan*D(r).mean()
        oG.zero_grad(); gl.backward(); oG.step()
        if s%2000==0: log(f"  [{tag}] {s}/{steps} G {gl.item():.4f} D {dl.item():.4f}")
    torch.save(m.state_dict(),ck); m.eval(); return m
if "grao" not in RES:
    mg=estagio2(); RES["grao"]=avaliar(mg,held,tag="grao"); save()
else:
    mg=PyramidN(BASE_SPECS).to(dev); mg.load_state_dict(torch.load(f"{OUT}/grao.pt")); mg.eval()
a=RES["lam1e-03"][-1]; g=RES["grao"][-1]
print("=== OS DOIS EIXOS (mesma taxa de bits) ===")
print(f"  MSE puro  : PSNR {a['psnr']:.2f} · nitidez {a['sharp']:.5f} @ {a['kb']:.2f} KB")
print(f"  +grão     : PSNR {g['psnr']:.2f} · nitidez {g['sharp']:.5f} @ {g['kb']:.2f} KB")
print(f"  → PSNR {g['psnr']-a['psnr']:+.2f} dB · nitidez {100*(g['sharp']/max(a['sharp'],1e-9)-1):+.0f}%")


## 9) Comparação VISUAL (é aqui que se decide)


In [ ]:
m0=PyramidN(BASE_SPECS).to(dev); m0.load_state_dict(torch.load(f"{OUT}/lam1e-03.pt")); m0.eval()
for idx in [0,3,7]:
    im=held[idx]; x=im[None].to(dev)
    with torch.no_grad():
        r0=m0(x,train=False); rg=mg(x,train=False)
        kb=sum(kb_of(z) for z in r0["z"])
    jq=None; d=baselines(im,idx)
    jj=min(d["JPEG"],key=lambda t:abs(t[0]-kb))
    arr=(im.permute(1,2,0).numpy()*255).astype(np.uint8)
    for q in range(2,98,3):
        b=io.BytesIO(); Image.fromarray(arr).save(b,"JPEG",quality=q)
        if abs(len(b.getvalue())/1024-kb)<0.4: jq=np.asarray(Image.open(io.BytesIO(b.getvalue())).convert("RGB"))/255.; break
    fig,ax=plt.subplots(1,4,figsize=(18,4.6))
    ax[0].imshow(im.permute(1,2,0)); ax[0].set_title("ORIGINAL")
    ax[1].imshow(r0["rec"][0].cpu().permute(1,2,0).clamp(0,1)); ax[1].set_title(f"IARA MSE {psnr(r0['rec'][0].cpu().numpy(),im.numpy()):.1f}dB / {kb:.1f}KB")
    ax[2].imshow(rg["rec"][0].cpu().permute(1,2,0).clamp(0,1)); ax[2].set_title(f"IARA +GRÃO {psnr(rg['rec'][0].cpu().numpy(),im.numpy()):.1f}dB / {kb:.1f}KB")
    if jq is not None: ax[3].imshow(jq); ax[3].set_title(f"JPEG {jj[1]:.1f}dB (mesma banda)")
    for a_ in ax: a_.axis("off")
    plt.tight_layout(); plt.show()


## 10) EXP-5 — domínio estreito vs diverso + RESUMO FINAL


In [ ]:
def feat(img):
    x=F.interpolate(img[None].cpu(),size=(S,S),mode="bilinear")[0]
    h=torch.cat([torch.histc(x[c],bins=4,min=0,max=1) for c in range(3)])/(S*S)
    g=x.mean(0); ft=torch.fft.rfft2(g).abs(); H,W=ft.shape
    yy,xx=torch.meshgrid(torch.arange(H),torch.arange(W),indexing="ij"); rr=torch.sqrt((yy/H)**2+(xx/W)**2)
    bands=torch.tensor([ft[(rr>=a)&(rr<b)].mean() for a,b in [(0,.05),(.05,.15),(.15,.3),(.3,.5),(.5,1.)]])
    return torch.cat([h,bands/(bands.sum()+1e-9)]).numpy()
if "estreito" not in RES:
    Ftr=np.stack([feat(i) for i in pool]); mu,sd=Ftr.mean(0),Ftr.std(0)+1e-9; Fn=(Ftr-mu)/sd
    K=4; rg_=np.random.default_rng(0); cent=Fn[rg_.choice(len(Fn),K,replace=False)].copy()
    for _ in range(50):
        lab=((Fn[:,None]-cent[None])**2).sum(-1).argmin(1)
        for k in range(K):
            if (lab==k).any(): cent[k]=Fn[lab==k].mean(0)
    big=int(np.bincount(lab).argmax()); idxN=[i for i in range(len(pool)) if lab[i]==big]
    hl=(np.stack([feat(i) for i in held])-mu)/sd; hlab=((hl[:,None]-cent[None])**2).sum(-1).argmin(1)
    held_n=[held[i] for i in range(len(held)) if hlab[i]==big]
    log(f"estreito: {len(idxN)} treino / {len(held_n)} held-out")
    mn=treinar(BEST,1.2e-3,idxs=idxN,tag="estreito")
    RES["estreito"]=avaliar(mn,held_n,n=min(24,len(held_n)),tag="estreito")
    RES["diverso_no_estreito"]=avaliar(m0,held_n,n=min(24,len(held_n)),tag="diverso_no_estreito"); save()
print("\n"+"="*78); print("RESUMO FINAL — nível mais fino de cada experimento"); print("="*78)
hdr=f"{'experimento':22s} {'PSNR':>7s} {'KB':>7s} {'ΔJPEG':>7s} {'ΔWebP':>7s} {'ΔAVIF':>7s} {'nitidez':>9s}"
print(hdr); print("-"*78)
for k in RES:
    f=RES[k][-1]
    dj=f["psnr"]-f.get("JPEG",np.nan); dw=f["psnr"]-f.get("WebP",np.nan); da=f["psnr"]-f.get("AVIF",np.nan)
    print(f"{k:22s} {f['psnr']:7.2f} {f['kb']:7.2f} {dj:+7.2f} {dw:+7.2f} {da:+7.2f} {f['sharp']:9.5f}")
print("-"*78); print(f"PNG sem perdas: {PNG_KB:.1f} KB · resultados em {OUT}/resultados.json")
